# Interactive ATM IV Viewer

Interactive visualization of ATM Implied Volatility time series with earnings detection signals.

Features:
- Zoom and pan on charts
- Hover to see exact values
- Multiple maturities (30d, 60d, 90d)
- Earnings signals (IV spikes and crushes)
- Spot price overlay
- Comparison between EOD and Minute-Aligned methods

In [ ]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime, timedelta
from pathlib import Path
import subprocess
import os

## Configuration

Set your parameters here:

In [ ]:
# ============= CONFIGURATION =============

SYMBOL = "AAPL"
START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
METHOD = "minute-aligned"  # Options: "eod" or "minute-aligned"

# Data directory (will use FINQ_DATA_DIR env var if set)
DATA_DIR = os.getenv("FINQ_DATA_DIR", str(Path.home() / "polygon/data"))

# Detection thresholds
IV_SPIKE_THRESHOLD = 0.20  # 20% increase
IV_CRUSH_THRESHOLD = 0.15  # 15% decrease

# ==========================================

## Generate IV Data

Run the Rust CLI to generate IV time series data:

In [ ]:
output_dir = Path(f"./iv_data_{METHOD.replace('-', '_')}")
output_dir.mkdir(exist_ok=True)

# Build command
cmd = [
    "./target/release/cs", "atm-iv",
    "--symbols", SYMBOL,
    "--start", START_DATE,
    "--end", END_DATE,
    "--output", str(output_dir),
]

if METHOD == "minute-aligned":
    cmd.append("--minute-aligned")

# Run command
print(f"Generating {METHOD} IV data for {SYMBOL} from {START_DATE} to {END_DATE}...")
env = os.environ.copy()
env["FINQ_DATA_DIR"] = DATA_DIR

result = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("Error:", result.stderr)
    raise RuntimeError(f"Command failed with return code {result.returncode}")

print("✓ Data generation complete")

## Load and Prepare Data

In [ ]:
# Load data
data_file = output_dir / f"atm_iv_{SYMBOL}.parquet"
df = pl.read_parquet(data_file)

# Convert date from days since epoch to datetime
epoch = datetime(1970, 1, 1)
df = df.with_columns([
    pl.col('date').map_elements(
        lambda d: epoch + timedelta(days=d), 
        return_dtype=pl.Datetime
    ).alias('datetime')
])

# Convert IVs to percentages
df = df.with_columns([
    (pl.col('atm_iv_30d') * 100).alias('iv_30d_pct'),
    (pl.col('atm_iv_60d') * 100).alias('iv_60d_pct'),
    (pl.col('atm_iv_90d') * 100).alias('iv_90d_pct'),
])

print(f"Loaded {len(df)} observations")
print(f"Date range: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"\nFirst few rows:")
df.head()

## Detect Earnings Signals

In [ ]:
# Calculate day-over-day changes
df = df.sort('datetime')
df = df.with_columns([
    pl.col('atm_iv_30d').pct_change().alias('iv_30d_change'),
])

# Detect spikes and crushes
df = df.with_columns([
    (pl.col('iv_30d_change') > IV_SPIKE_THRESHOLD).alias('is_spike'),
    (pl.col('iv_30d_change') < -IV_CRUSH_THRESHOLD).alias('is_crush'),
])

# Extract signal events
spikes = df.filter(pl.col('is_spike')).select(['datetime', 'iv_30d_pct', 'iv_30d_change'])
crushes = df.filter(pl.col('is_crush')).select(['datetime', 'iv_30d_pct', 'iv_30d_change'])

print(f"Detected {len(spikes)} IV spikes (>{IV_SPIKE_THRESHOLD*100:.0f}%)")
print(f"Detected {len(crushes)} IV crushes (>{IV_CRUSH_THRESHOLD*100:.0f}%)")

if len(spikes) > 0:
    print("\nIV Spikes:")
    print(spikes.to_pandas())

if len(crushes) > 0:
    print("\nIV Crushes:")
    print(crushes.to_pandas())

## Interactive Chart: IV Time Series

In [ ]:
# Convert to pandas for plotly
df_pd = df.to_pandas()

# Create figure with secondary y-axis
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        f'{SYMBOL} - ATM Implied Volatility ({METHOD.upper()})',
        f'{SYMBOL} - Spot Price'
    ),
    row_heights=[0.7, 0.3],
)

# Add 30d IV line
fig.add_trace(
    go.Scatter(
        x=df_pd['datetime'],
        y=df_pd['iv_30d_pct'],
        name='30d IV',
        line=dict(color='#1f77b4', width=2),
        hovertemplate='<b>30d IV</b><br>' +
                      'Date: %{x|%Y-%m-%d}<br>' +
                      'IV: %{y:.2f}%<br>' +
                      '<extra></extra>'
    ),
    row=1, col=1
)

# Add 60d IV line (if available)
if df_pd['iv_60d_pct'].notna().any():
    fig.add_trace(
        go.Scatter(
            x=df_pd[df_pd['iv_60d_pct'].notna()]['datetime'],
            y=df_pd[df_pd['iv_60d_pct'].notna()]['iv_60d_pct'],
            name='60d IV',
            line=dict(color='#ff7f0e', width=2),
            hovertemplate='<b>60d IV</b><br>' +
                          'Date: %{x|%Y-%m-%d}<br>' +
                          'IV: %{y:.2f}%<br>' +
                          '<extra></extra>'
        ),
        row=1, col=1
    )

# Add 90d IV line (if available)
if df_pd['iv_90d_pct'].notna().any():
    fig.add_trace(
        go.Scatter(
            x=df_pd[df_pd['iv_90d_pct'].notna()]['datetime'],
            y=df_pd[df_pd['iv_90d_pct'].notna()]['iv_90d_pct'],
            name='90d IV',
            line=dict(color='#2ca02c', width=2),
            hovertemplate='<b>90d IV</b><br>' +
                          'Date: %{x|%Y-%m-%d}<br>' +
                          'IV: %{y:.2f}%<br>' +
                          '<extra></extra>'
        ),
        row=1, col=1
    )

# Add IV spikes
if len(spikes) > 0:
    spikes_pd = spikes.to_pandas()
    fig.add_trace(
        go.Scatter(
            x=spikes_pd['datetime'],
            y=spikes_pd['iv_30d_pct'],
            mode='markers',
            name='IV Spike',
            marker=dict(color='yellow', size=12, symbol='star',
                       line=dict(color='orange', width=2)),
            hovertemplate='<b>IV Spike</b><br>' +
                          'Date: %{x|%Y-%m-%d}<br>' +
                          'IV: %{y:.2f}%<br>' +
                          'Change: %{customdata:.1f}%<br>' +
                          '<extra></extra>',
            customdata=spikes_pd['iv_30d_change'] * 100
        ),
        row=1, col=1
    )

# Add IV crushes
if len(crushes) > 0:
    crushes_pd = crushes.to_pandas()
    fig.add_trace(
        go.Scatter(
            x=crushes_pd['datetime'],
            y=crushes_pd['iv_30d_pct'],
            mode='markers',
            name='IV Crush',
            marker=dict(color='red', size=10, symbol='circle'),
            hovertemplate='<b>IV Crush</b><br>' +
                          'Date: %{x|%Y-%m-%d}<br>' +
                          'IV: %{y:.2f}%<br>' +
                          'Change: %{customdata:.1f}%<br>' +
                          '<extra></extra>',
            customdata=crushes_pd['iv_30d_change'] * 100
        ),
        row=1, col=1
    )

# Add spot price
fig.add_trace(
    go.Scatter(
        x=df_pd['datetime'],
        y=df_pd['spot'],
        name='Spot Price',
        line=dict(color='black', width=1.5),
        hovertemplate='<b>Spot Price</b><br>' +
                      'Date: %{x|%Y-%m-%d}<br>' +
                      'Price: $%{y:.2f}<br>' +
                      '<extra></extra>'
    ),
    row=2, col=1
)

# Update layout
fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_yaxes(title_text="Implied Volatility (%)", row=1, col=1)
fig.update_yaxes(title_text="Price ($)", row=2, col=1)

fig.update_layout(
    height=800,
    hovermode='x unified',
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

## Interactive Chart: IV Changes (Day-over-Day)

In [ ]:
fig2 = go.Figure()

# Add daily change as bar chart
colors = ['red' if x < 0 else 'green' for x in df_pd['iv_30d_change']]
fig2.add_trace(
    go.Bar(
        x=df_pd['datetime'],
        y=df_pd['iv_30d_change'] * 100,
        name='Daily Change',
        marker_color=colors,
        hovertemplate='<b>Daily IV Change</b><br>' +
                      'Date: %{x|%Y-%m-%d}<br>' +
                      'Change: %{y:.2f}%<br>' +
                      '<extra></extra>'
    )
)

# Add threshold lines
fig2.add_hline(
    y=IV_SPIKE_THRESHOLD * 100, 
    line_dash="dash", 
    line_color="orange",
    annotation_text=f"Spike Threshold ({IV_SPIKE_THRESHOLD*100:.0f}%)",
    annotation_position="right"
)
fig2.add_hline(
    y=-IV_CRUSH_THRESHOLD * 100, 
    line_dash="dash", 
    line_color="red",
    annotation_text=f"Crush Threshold (-{IV_CRUSH_THRESHOLD*100:.0f}%)",
    annotation_position="right"
)

fig2.update_layout(
    title=f'{SYMBOL} - Daily IV Change (30d)',
    xaxis_title='Date',
    yaxis_title='Daily Change (%)',
    height=500,
    hovermode='x unified',
    showlegend=False
)

fig2.show()

## Summary Statistics

In [ ]:
print(f"{'='*60}")
print(f"ATM IV Statistics - {SYMBOL} ({METHOD.upper()})")
print(f"{START_DATE} to {END_DATE}")
print(f"{'='*60}\n")

print("30-Day ATM IV:")
print(f"  Observations: {df['atm_iv_30d'].drop_nulls().len()}")
print(f"  Mean: {df['atm_iv_30d'].mean()*100:.2f}%")
print(f"  Median: {df['atm_iv_30d'].median()*100:.2f}%")
print(f"  Std Dev: {df['atm_iv_30d'].std()*100:.2f}%")
print(f"  Min: {df['atm_iv_30d'].min()*100:.2f}%")
print(f"  Max: {df['atm_iv_30d'].max()*100:.2f}%")

if df['atm_iv_60d'].drop_nulls().len() > 0:
    print("\n60-Day ATM IV:")
    print(f"  Observations: {df['atm_iv_60d'].drop_nulls().len()}")
    print(f"  Mean: {df['atm_iv_60d'].mean()*100:.2f}%")
    print(f"  Median: {df['atm_iv_60d'].median()*100:.2f}%")

if df['atm_iv_90d'].drop_nulls().len() > 0:
    print("\n90-Day ATM IV:")
    print(f"  Observations: {df['atm_iv_90d'].drop_nulls().len()}")
    print(f"  Mean: {df['atm_iv_90d'].mean()*100:.2f}%")
    print(f"  Median: {df['atm_iv_90d'].median()*100:.2f}%")

print("\nEarnings Signals:")
print(f"  IV Spikes (>{IV_SPIKE_THRESHOLD*100:.0f}%): {len(spikes)}")
print(f"  IV Crushes (>{IV_CRUSH_THRESHOLD*100:.0f}%): {len(crushes)}")

print(f"\n{'='*60}")

## Term Structure Analysis

Visualize the term structure (relationship between different maturities):

In [ ]:
# Filter to dates where we have all three maturities
df_complete = df.filter(
    pl.col('atm_iv_30d').is_not_null() & 
    pl.col('atm_iv_60d').is_not_null() & 
    pl.col('atm_iv_90d').is_not_null()
)

if len(df_complete) > 0:
    # Calculate term spreads
    df_complete = df_complete.with_columns([
        ((pl.col('atm_iv_30d') - pl.col('atm_iv_60d')) * 100).alias('spread_30_60'),
        ((pl.col('atm_iv_30d') - pl.col('atm_iv_90d')) * 100).alias('spread_30_90'),
    ])
    
    df_complete_pd = df_complete.to_pandas()
    
    fig3 = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=(
            'Term Structure: All Maturities',
            'Term Spread (30d - 60d)'
        ),
        row_heights=[0.6, 0.4],
    )
    
    # Plot all maturities
    fig3.add_trace(
        go.Scatter(x=df_complete_pd['datetime'], y=df_complete_pd['iv_30d_pct'],
                   name='30d', line=dict(color='#1f77b4')),
        row=1, col=1
    )
    fig3.add_trace(
        go.Scatter(x=df_complete_pd['datetime'], y=df_complete_pd['iv_60d_pct'],
                   name='60d', line=dict(color='#ff7f0e')),
        row=1, col=1
    )
    fig3.add_trace(
        go.Scatter(x=df_complete_pd['datetime'], y=df_complete_pd['iv_90d_pct'],
                   name='90d', line=dict(color='#2ca02c')),
        row=1, col=1
    )
    
    # Plot term spread
    fig3.add_trace(
        go.Scatter(x=df_complete_pd['datetime'], y=df_complete_pd['spread_30_60'],
                   name='30d - 60d', line=dict(color='purple'),
                   fill='tozeroy'),
        row=2, col=1
    )
    
    # Add zero line to spread chart
    fig3.add_hline(y=0, line_dash="dash", line_color="black", row=2, col=1)
    
    fig3.update_yaxes(title_text="IV (%)", row=1, col=1)
    fig3.update_yaxes(title_text="Spread (%)", row=2, col=1)
    fig3.update_xaxes(title_text="Date", row=2, col=1)
    
    fig3.update_layout(
        height=700,
        hovermode='x unified',
        title_text=f'{SYMBOL} - Term Structure Analysis'
    )
    
    fig3.show()
    
    # Statistics
    print("Term Structure Statistics:")
    print(f"  Days with complete data: {len(df_complete)}")
    print(f"  Avg 30d-60d spread: {df_complete['spread_30_60'].mean():.2f}%")
    print(f"  Avg 30d-90d spread: {df_complete['spread_30_90'].mean():.2f}%")
    print(f"  Backwardation days (30d > 60d): {(df_complete['spread_30_60'] > 0).sum()}")
else:
    print("Not enough data for term structure analysis (need 30d, 60d, and 90d IVs)")

## Export Data

Save the processed data with signals for further analysis:

In [ ]:
# Export to CSV for easy viewing
export_path = output_dir / f"{SYMBOL}_iv_with_signals.csv"
df.select([
    'datetime', 'spot', 
    'iv_30d_pct', 'iv_60d_pct', 'iv_90d_pct',
    'iv_30d_change', 'is_spike', 'is_crush'
]).write_csv(export_path)

print(f"✓ Exported data to: {export_path}")